# Handwritten Digit Recognition Using Convolutional Neural Networks

**Course:** CS5720 Neural Network and Deep Learning (Spring 2025)  
**University:** University of Central Missouri  

**Team Members:**
- Lokesh Reddy Siripireddy (700758270)
- Sony Pailla (70076544)
- Mundru Satya Sri Lasya (700758682)
- Jasmitha Nalla (700799225)

---

### Table of Contents
1. [Import Libraries](#1)
2. [Load & Preprocess Data](#2)
3. [Build CNN Model](#3)
4. [Compile & Train](#4)
5. [Evaluate & Confusion Matrix](#5)
6. [Visualize Predictions](#6)

---
## 1. Import Libraries <a id='1'></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

---
## 2. Load & Preprocess Data <a id='2'></a>

The **MNIST dataset** contains 70,000 grayscale images (28×28 pixels) of handwritten digits (0–9):
- 60,000 training images
- 10,000 test images

Preprocessing steps:
- **Normalize** pixel values from [0, 255] → [0, 1] for faster convergence
- **Reshape** to (28, 28, 1) to add channel dimension required by CNN
- **One-hot encode** labels for categorical classification

In [ ]:
# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f"Training samples : {x_train.shape[0]}")
print(f"Test samples     : {x_test.shape[0]}")
print(f"Image size       : {x_train.shape[1]}x{x_train.shape[2]} pixels")

In [ ]:
# Normalize pixel values to [0, 1]
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# Reshape to (samples, 28, 28, 1) — CNNs require 4D input
x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

# One-hot encode labels (e.g., digit 3 → [0,0,0,1,0,0,0,0,0,0])
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train_cat.shape)

In [ ]:
# Visualize sample digits from the dataset
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    idx = np.where(y_train == i)[0][0]
    axes[0, i].imshow(x_train[idx].reshape(28, 28), cmap='gray')
    axes[0, i].set_title(f'Digit {i}', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(x_train[idx + 1].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')

plt.suptitle('Sample MNIST Images (Digits 0–9)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Build CNN Architecture <a id='3'></a>

| Layer | Details | Output Shape |
|-------|---------|-------------|
| Input | Grayscale image | (28, 28, 1) |
| Conv2D | 32 filters, 3×3, ReLU | (26, 26, 32) |
| MaxPooling2D | 2×2 | (13, 13, 32) |
| Conv2D | 64 filters, 3×3, ReLU | (11, 11, 64) |
| MaxPooling2D | 2×2 | (5, 5, 64) |
| Flatten | — | (1600,) |
| Dense | 128 units, ReLU | (128,) |
| Dropout | rate=0.5 | (128,) |
| Output (Dense) | 10 units, Softmax | (10,) |

In [ ]:
model = Sequential([
    # First Convolutional Block
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    # Second Convolutional Block
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Classifier Head
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')   # 10 output classes (digits 0–9)
])

model.summary()

---
## 4. Compile & Train <a id='4'></a>

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(x_test, y_test_cat)
)

In [ ]:
# Plot Training vs Validation Accuracy & Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2, linestyle='--')
axes[0].set_title('Model Accuracy over Epochs', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--')
axes[1].set_title('Model Loss over Epochs', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Evaluate & Confusion Matrix <a id='5'></a>

In [ ]:
# Final evaluation on test set
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test Accuracy : {test_acc * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

In [ ]:
# Generate predictions
y_pred_prob = model.predict(x_test)
y_pred      = np.argmax(y_pred_prob, axis=1)

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix — CNN Digit Classifier', fontsize=14)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

---
## 6. Visualize Predictions <a id='6'></a>

In [ ]:
# Show correct and misclassified predictions
correct_idx   = np.where(y_pred == y_test)[0][:10]
incorrect_idx = np.where(y_pred != y_test)[0][:10]

fig, axes = plt.subplots(2, 10, figsize=(20, 5))

for i, idx in enumerate(correct_idx):
    axes[0, i].imshow(x_test[idx].reshape(28, 28), cmap='gray')
    axes[0, i].set_title(f'✓ {y_pred[idx]}', color='green', fontsize=10)
    axes[0, i].axis('off')

for i, idx in enumerate(incorrect_idx):
    axes[1, i].imshow(x_test[idx].reshape(28, 28), cmap='gray')
    axes[1, i].set_title(f'✗ P:{y_pred[idx]} A:{y_test[idx]}', color='red', fontsize=8)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Correct', fontsize=11)
axes[1, 0].set_ylabel('Misclassified', fontsize=11)
plt.suptitle('CNN Predictions: Correct vs Misclassified', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()